# Land Use prediction

In [ ]:
import pandas as pd
blocks_gdf = pd.read_pickle('./../../data/saint_petersburg/blocks.pickle')

In [ ]:
from blocksnet.analysis.land_use.prediction import SpatialClassifier

In [ ]:
classifier = SpatialClassifier.default()
test_ready = classifier.preprocess_run_data(blocks_gdf)
result = classifier.run(test_ready)

In [3]:
result

,geometry,category,pred_name,prob_RESIDENTIAL,prob_BUSINESS,prob_RECREATION,prob_INDUSTRIAL
0,"MULTIPOLYGON (((349424.859 6631180.891, 349424...",LandUseCategory.INDUSTRIAL,INDUSTRIAL,0.045165,0.028159,0.305649,0.621027
1,"MULTIPOLYGON (((352083.617 6633950.146, 352240...",LandUseCategory.RECREATION,RESIDENTIAL,0.376333,0.209277,0.238064,0.176327
2,"MULTIPOLYGON (((346700.642 6618453.176, 346681...",LandUseCategory.RESIDENTIAL,RESIDENTIAL,0.724700,0.118614,0.118224,0.038462
3,"MULTIPOLYGON (((347043.363 6618261.219, 347042...",LandUseCategory.RESIDENTIAL,RECREATION,0.256970,0.138033,0.451670,0.153327
4,"MULTIPOLYGON (((354879.039 6618859.116, 354845...",LandUseCategory.RESIDENTIAL,RESIDENTIAL,0.579051,0.133522,0.208631,0.078796
...,...,...,...,...,...,...,...
9244,"MULTIPOLYGON (((346635.461 6647492.048, 346473...",LandUseCategory.RESIDENTIAL,RECREATION,0.123647,0.019748,0.839957,0.016648
9245,"MULTIPOLYGON (((346361.221 6647603.446, 346328...",LandUseCategory.INDUSTRIAL,BUSINESS,0.163405,0.432571,0.302783,0.101240
9246,"MULTIPOLYGON (((344109.285 6649134.367, 344000...",LandUseCategory.RECREATION,RECREATION,0.296557,0.160168,0.402465,0.140809
9247,"MULTIPOLYGON (((346323.488 6649497.386, 346199...",LandUseCategory.INDUSTRIAL,RESIDENTIAL,0.479835,0.183405,0.296642,0.040117


# Land Use Train Mode

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

from blocksnet.analysis.land_use.prediction import SpatialClassifier
from blocksnet.machine_learning.strategy.sklearn.ensemble.voting.classification_strategy import SKLearnVotingClassificationStrategy

In [ ]:
import pandas as pd
blocks_gdf = pd.read_pickle('./../../data/saint_petersburg/blocks.pickle')


In [4]:
estimators = [
    ('XGB', XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.3,
        eval_metric='mlogloss',
        use_label_encoder=False,
        n_jobs=-1,
        random_state=42
    )),
    ('LGBM', LGBMClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.3,
        class_weight='balanced',
        verbose=-1,
        n_jobs=-1,
        random_state=42
    )),
    ('RF', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )),
]

strategy = SKLearnVotingClassificationStrategy(estimators, {"voting": "soft", "n_jobs": -1})
classifier = SpatialClassifier(strategy, 1000, 5)


In [ ]:
train_ready = classifier.preprocess_training_data(blocks_gdf)


In [ ]:
classifier.train(train_ready)

In [ ]:
raw_test = pd.read_pickle('./../../data/saint_petersburg/test_blocks.pickle')
test_ready = classifier.preprocess_run_data(raw_test)


In [ ]:
result = classifier.run(test_ready)


In [ ]:
result


,geometry,category,pred_name,prob_RESIDENTIAL,prob_BUSINESS,prob_RECREATION,prob_INDUSTRIAL
0,"MULTIPOLYGON (((349424.859 6631180.891, 349424...",LandUseCategory.INDUSTRIAL,INDUSTRIAL,0.045165,0.028159,0.305649,0.621027
1,"MULTIPOLYGON (((352083.617 6633950.146, 352240...",LandUseCategory.RECREATION,RESIDENTIAL,0.376333,0.209277,0.238064,0.176327
2,"MULTIPOLYGON (((346700.642 6618453.176, 346681...",LandUseCategory.RESIDENTIAL,RESIDENTIAL,0.724700,0.118614,0.118224,0.038462
3,"MULTIPOLYGON (((347043.363 6618261.219, 347042...",LandUseCategory.RESIDENTIAL,RECREATION,0.256970,0.138033,0.451670,0.153327
4,"MULTIPOLYGON (((354879.039 6618859.116, 354845...",LandUseCategory.RESIDENTIAL,RESIDENTIAL,0.579051,0.133522,0.208631,0.078796
...,...,...,...,...,...,...,...
9244,"MULTIPOLYGON (((346635.461 6647492.048, 346473...",LandUseCategory.RESIDENTIAL,RECREATION,0.123647,0.019748,0.839957,0.016648
9245,"MULTIPOLYGON (((346361.221 6647603.446, 346328...",LandUseCategory.INDUSTRIAL,BUSINESS,0.163405,0.432571,0.302783,0.101240
9246,"MULTIPOLYGON (((344109.285 6649134.367, 344000...",LandUseCategory.RECREATION,RECREATION,0.296557,0.160168,0.402465,0.140809
9247,"MULTIPOLYGON (((346323.488 6649497.386, 346199...",LandUseCategory.INDUSTRIAL,RESIDENTIAL,0.479835,0.183405,0.296642,0.040117


In [ ]:
classifier.save('artifacts')
